# 📅 Lab W4-3 — Time Intelligence ด้วย Window Functions

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 4 — OLAP and Multidimensional Analysis**

Lab นี้ใช้คู่กับสื่อจำลอง **Time Intelligence Builder** (`/sims/time-intelligence`)
ตัวเลขที่คุณคำนวณได้ในสมุดเล่มนี้ต้องตรงกับตัวเลขบนหน้าจอสื่อจำลองทุกหลัก

## สิ่งที่จะได้เรียนรู้
1. เขียน **YTD · MoM · YoY · Rolling 12 เดือน** ด้วย window function ได้ถูกต้อง
2. อธิบายว่าเหตุใดช่วงที่คำนวณไม่ได้ต้องเป็น **NaN ไม่ใช่ 0**
3. แสดงให้เห็นว่า **การเลือกตัววัดคือการเลือกว่าจะให้ผู้บริหารเห็นอะไร**
4. ตรวจจับ **ผลกระทบย้อนกลับ (rebound artifact)** ที่ YoY สร้างขึ้นหลังเหตุการณ์ผิดปกติ

## ข้อมูล
`sales_3years_daily.csv` — ยอดขายรายวันรายสาขา 3 ปีเต็ม (2023–2025)
มีเหตุการณ์ผิดปกติซ่อนอยู่ 1 เหตุการณ์ที่ยังไม่มีใครบอกคุณ

In [ ]:
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week04/sales_3years_daily.csv")
raw = pd.read_csv(URL)

print(f"จำนวนแถวรายวัน : {len(raw):,}")
print(f"ช่วงวันที่      : {raw.sales_date.min()} ถึง {raw.sales_date.max()}")
print(f"สาขา            : {sorted(raw.store_id.unique())}")
print(f"ภูมิภาค         : {sorted(raw.region.unique())}")
raw.head(5)

## ส่วนที่ 1 — ยุบให้เป็นระดับเดือน

ตัววัดเชิงเวลาทุกตัวคำนวณบน **grain ระดับเดือน**
ถ้ายุบผิดตั้งแต่ขั้นนี้ ตัววัดทั้ง 5 ตัวจะผิดตามทั้งหมด

### 🧑‍💻 งานที่ 1
สร้าง `DataFrame` ชื่อ `m` ที่มีหนึ่งแถวต่อหนึ่งเดือน มีคอลัมน์ `amount`
โดย **เรียงตามเวลาจากน้อยไปมาก** และมี index เป็นสตริง `YYYY-MM`

แล้วตรวจว่าได้ครบ 36 เดือนจริง และผลรวมเท่ากับผลรวมของไฟล์ดิบ

*เฉลยที่ถูกต้อง: 36 เดือน · ผลรวม 414,550,351.10 บาท*

In [ ]:
raw["ym"] = raw.sales_date.str[:7]
m = raw.groupby("ym").net_amount.sum().sort_index().to_frame("amount")
m["year"] = m.index.str[:4].astype(int)

print(f"จำนวนเดือน : {len(m)}")
print(f"เดือนแรก   : {m.index[0]}   เดือนสุดท้าย : {m.index[-1]}")
print(f"ผลรวมรายเดือน : {m.amount.sum():,.2f}")
print(f"ผลรวมไฟล์ดิบ  : {raw.net_amount.sum():,.2f}")
assert abs(m.amount.sum() - raw.net_amount.sum()) < 0.01
assert len(m) == 36 and m.index.is_monotonic_increasing
print("✓ ยุบระดับเดือนถูกต้องและเรียงตามเวลาแล้ว")

> **กับดักข้อแรก** window function ทุกตัวสมมติว่าแถวเรียงตามเวลาแล้ว
> ถ้าลืม `sort_index()` ค่า `shift()` และ `rolling()` จะเลื่อนไปผิดเดือนโดยไม่มี error ใด ๆ
> — ผลลัพธ์ผิดเงียบ ๆ ซึ่งอันตรายกว่าโปรแกรมล้ม

## ส่วนที่ 2 — ตัววัดเชิงเวลาทั้ง 5 ตัว

### 🧑‍💻 งานที่ 2
เพิ่มคอลัมน์ต่อไปนี้ลงใน `m`

| คอลัมน์ | ความหมาย | ข้อควรระวัง |
|---|---|---|
| `ytd` | ยอดสะสมตั้งแต่ต้นปี | ต้อง **รีเซ็ตทุกวันที่ 1 มกราคม** |
| `mom` | %เทียบเดือนก่อนหน้า | เดือนแรกคำนวณไม่ได้ |
| `yoy` | %เทียบเดือนเดียวกันปีก่อน | 12 เดือนแรกคำนวณไม่ได้ |
| `roll12` | ผลรวมเคลื่อนที่ 12 เดือน | 11 เดือนแรกคำนวณไม่ได้ |

**ห้ามเติม 0 ในช่องที่คำนวณไม่ได้** ให้เป็น `NaN`

*เฉลยที่ถูกต้อง: YTD ของ 2023-12 = 129,644,557.50 · YoY ของ 2024-01 = +7.49%*

In [ ]:
m["ytd"] = m.groupby("year").amount.cumsum()
m["mom"] = m.amount.pct_change() * 100
m["yoy"] = m.amount.pct_change(12) * 100
m["roll12"] = m.amount.rolling(12).sum()

print(m.round(2).to_string())

print("\nจำนวนช่องที่เป็น NaN (คำนวณไม่ได้จริง)")
print(m[["mom", "yoy", "roll12"]].isna().sum().to_string())

> **ตรวจความถูกต้องของ YTD** ค่าสุดท้ายของแต่ละปีต้องเท่ากับยอดรวมทั้งปีพอดี

In [ ]:
check = m.groupby("year").agg(ytd_last=("ytd", "last"), year_total=("amount", "sum"))
check["ตรงกัน"] = (check.ytd_last - check.year_total).abs() < 0.01
print(check.to_string())
assert check["ตรงกัน"].all(), "YTD ไม่รีเซ็ตเมื่อขึ้นปีใหม่"
print("\n✓ YTD รีเซ็ตทุกต้นปีถูกต้อง")

## ส่วนที่ 3 — เหตุการณ์ที่ซ่อนอยู่

ตอนนี้ให้หาเองว่ามีอะไรผิดปกติ โดยยังไม่ต้องอ่านคำเฉลย

### 🧑‍💻 งานที่ 3
ใช้ตัววัดที่คำนวณไว้ หา **เดือนที่ผิดปกติที่สุด** แล้วเจาะลงไปถึงระดับวัน
เพื่อระบุว่าเหตุการณ์เริ่มและจบวันที่เท่าไร

แนวทาง: เดือนที่ `yoy` ต่ำที่สุด → แล้ว plot หรือ print ยอดรายวันของเดือนนั้น

*เฉลยที่ถูกต้อง: มิถุนายน 2024 · ระบบขายขัดข้องระหว่างวันที่ 5–18*

In [ ]:
worst = m.yoy.idxmin()
print(f"เดือนที่ YoY ต่ำที่สุด : {worst}  ({m.loc[worst,'yoy']:.2f}%)")
print(f"ยอดของเดือนนั้น        : {m.loc[worst,'amount']:,.2f} บาท")

daily = raw[raw.ym == worst].groupby("sales_date").net_amount.sum()
normal = daily[daily > daily.median()].mean()

print(f"\nยอดรายวันของ {worst}")
for d, v in daily.items():
    flag = "  ← ต่ำผิดปกติ" if v < normal * 0.6 else ""
    print(f"  {d}  {v:>12,.2f}{flag}")

outage = daily[daily < normal * 0.6]
print(f"\nช่วงที่ผิดปกติ : {outage.index.min()} ถึง {outage.index.max()}  ({len(outage)} วัน)")
print(f"ยอดเฉลี่ยช่วงปกติ   : {normal:,.2f} บาท/วัน")
print(f"ยอดเฉลี่ยช่วงขัดข้อง : {outage.mean():,.2f} บาท/วัน "
      f"({outage.mean()/normal*100:.1f}% ของปกติ)")

## ส่วนที่ 4 — ตัววัดคนละตัว เล่าเรื่องคนละเรื่อง

นี่คือหัวใจของ Lab นี้ — เหตุการณ์เดียวกัน ข้อมูลชุดเดียวกัน
แต่ตัววัดที่เลือกทำให้มัน **เด่นหรือหายไป**

### 🧑‍💻 งานที่ 4
สร้างตารางเปรียบเทียบว่าเดือน 2024-06 หน้าตาเป็นอย่างไรภายใต้ตัววัดทั้ง 5 ตัว
โดยแสดงเป็น **%เบี่ยงเบนจากเดือนก่อนหน้า** ของตัววัดนั้น ๆ
เพื่อให้เทียบ "ความสะดุดตา" ข้ามตัววัดได้อย่างเป็นธรรม

แล้วตอบว่าตัววัดใดทำให้เหตุการณ์นี้ **มองไม่เห็น** และเพราะเหตุใด

In [ ]:
EVENT = "2024-06"
prev = m.index[m.index.get_loc(EVENT) - 1]

rows = []
for col, label in [("amount", "ยอดขายรายเดือน"), ("ytd", "YTD"),
                   ("roll12", "Rolling 12 เดือน")]:
    a, b = m.loc[prev, col], m.loc[EVENT, col]
    rows.append({"ตัววัด": label, f"ค่า {prev}": a, f"ค่า {EVENT}": b,
                 "เบี่ยงเบน %": (b / a - 1) * 100})

visibility = pd.DataFrame(rows).set_index("ตัววัด")
print(visibility.round(2).to_string())

print(f"\nMoM ของ {EVENT} : {m.loc[EVENT,'mom']:>8.2f}%")
print(f"YoY ของ {EVENT} : {m.loc[EVENT,'yoy']:>8.2f}%")

print("""
คำตอบ
-----
YTD และ Rolling 12 เดือน ทำให้เหตุการณ์นี้แทบมองไม่เห็น
  · YTD ยัง 'เพิ่มขึ้น' +16.66% เพราะเป็นยอดสะสมที่ไม่มีวันลดลง
    เดือนที่เสียหายที่สุดของปีจึงยังปรากฏเป็นแท่งที่สูงกว่าเดือนก่อนหน้า
  · Rolling 12 เดือน ลดลงเพียง −2.13% เพราะยอดที่หายไปถูกเฉลี่ยกับอีก 11 เดือนที่ปกติ

MoM และ YoY ทำให้เหตุการณ์เด่นชัด (−29.82% และ −22.36%)
เพราะทั้งสองตัวเป็น 'อัตราการเปลี่ยนแปลง' ที่เทียบกับฐานขนาดใกล้เคียงกัน

การเลือกตัววัดจึงเป็นการตัดสินใจเชิงบรรณาธิการ
ผู้ที่เลือกให้ dashboard แสดงเฉพาะ YTD กำลังเลือกให้ผู้บริหารไม่เห็นเหตุการณ์แบบนี้
""")

## ส่วนที่ 5 — กับดักที่อันตรายกว่า: ผลย้อนกลับของ YoY

### 🧑‍💻 งานที่ 5
ดูค่า `yoy` ของเดือน **มิถุนายน 2025** แล้วอธิบายว่าเหตุใดจึงสูงผิดปกติ
และเสนอวิธีแก้ที่ทำให้รายงานปี 2025 ไม่หลอกผู้อ่าน

*เฉลยที่ถูกต้อง: YoY ของ 2025-06 = +51.79% ทั้งที่ธุรกิจไม่ได้โตขนาดนั้น*

In [ ]:
print(f"YoY ของ 2025-06 : {m.loc['2025-06','yoy']:.2f}%")
print(f"YoY เฉลี่ยของเดือนอื่นในปี 2025 : "
      f"{m.loc[m.year == 2025, 'yoy'].drop('2025-06').mean():.2f}%")

print(f"\nยอด มิ.ย. 2023 : {m.loc['2023-06','amount']:>14,.2f}")
print(f"ยอด มิ.ย. 2024 : {m.loc['2024-06','amount']:>14,.2f}  ← ฐานที่ผิดปกติ")
print(f"ยอด มิ.ย. 2025 : {m.loc['2025-06','amount']:>14,.2f}")

# วิธีแก้ที่ 1 — เทียบกับปีที่ฐานยังปกติ (2 ปีย้อนหลัง)
two_yr = (m.loc["2025-06", "amount"] / m.loc["2023-06", "amount"] - 1) * 100
print(f"\nเทียบกับ มิ.ย. 2023 (2 ปี) : {two_yr:+.2f}%  "
      f"→ เฉลี่ยปีละ {((1+two_yr/100)**0.5 - 1)*100:+.2f}%")

# วิธีแก้ที่ 2 — ทำเครื่องหมายฐานที่ปนเปื้อนไว้ในตารางเลย
m["base_flag"] = ""
m.loc["2025-06", "base_flag"] = "⚠ ฐานปีก่อนได้รับผลจากเหตุขัดข้อง — ห้ามตีความว่าเป็นการเติบโต"
print(f"\n{m.loc['2025-06','base_flag']}")

print("""
วิธีแก้ที่ยอมรับได้ในทางปฏิบัติ
------------------------------
1. เก็บทะเบียนเหตุการณ์ผิดปกติ (event calendar) ไว้ใน dim_date
   แล้วให้รายงานติดธงอัตโนมัติเมื่อ 'ฐานเปรียบเทียบ' ตรงกับเหตุการณ์ในทะเบียน
2. เสริม YoY ด้วย CAGR 2 ปี หรือเทียบกับค่าปกติที่ปรับแล้ว (normalized baseline)
3. ห้ามลบหรือแก้ข้อมูลเดือนที่ขัดข้อง — เหตุการณ์นั้นเกิดขึ้นจริง
   สิ่งที่ต้องแก้คือ 'การตีความ' ไม่ใช่ 'ข้อมูล'
""")

## ส่วนที่ 6 — NaN ไม่ใช่ 0

### 🧑‍💻 งานที่ 6
สร้างสองเวอร์ชันของคอลัมน์ `yoy`

* `yoy_nan` — ปล่อยให้ 12 เดือนแรกเป็น `NaN` (ถูกต้อง)
* `yoy_zero` — เติม 0 แทน (ผิด แต่พบบ่อยมาก)

แล้วคำนวณ **ค่าเฉลี่ย YoY ตลอดช่วงข้อมูล** จากทั้งสองเวอร์ชัน
เพื่อแสดงว่าการเติม 0 บิดเบือนข้อสรุปไปกี่จุด

In [ ]:
yoy_nan = m.yoy
yoy_zero = m.yoy.fillna(0)

print(f"ค่าเฉลี่ย YoY (ปล่อย NaN) : {yoy_nan.mean():>7.2f}%   จาก {yoy_nan.notna().sum()} เดือนที่มีความหมาย")
print(f"ค่าเฉลี่ย YoY (เติม 0)    : {yoy_zero.mean():>7.2f}%   จาก {len(yoy_zero)} เดือน")
print(f"บิดเบือนไป                : {yoy_zero.mean() - yoy_nan.mean():>7.2f} จุดเปอร์เซ็นต์")

print("""
เหตุใดจึงร้ายแรง
----------------
0% ในภาษาของกราฟแปลว่า 'เติบโตเท่ากับปีก่อนพอดี' ซึ่งเป็นข้อความที่มีความหมาย
แต่ความจริงของ 12 เดือนแรกคือ 'ยังไม่มีปีก่อนให้เทียบ' ซึ่งไม่ใช่ข้อความเดียวกันเลย

กราฟ YoY ที่เติม 0 จะแสดงเส้นแบนสนิทตลอดปีแรก
ผู้อ่านจะสรุปว่า 'ธุรกิจนิ่งสนิททั้งปี' ทั้งที่ปีนั้นโตจาก 8.5 ล้านเป็น 10.6 ล้านบาท

ค่าว่างที่ถูกเติมผิดวิธีไม่ได้หายไปเฉย ๆ — มันกลายเป็นข้อมูลเท็จ
""")

## ส่วนที่ 7 — เขียนเป็น SQL

### 🧑‍💻 งานที่ 7 (เขียน SQL ไม่ต้องรัน)

เขียน SQL หนึ่งคำสั่งที่คืนตารางรายเดือนพร้อมทั้ง 4 ตัววัด
โดยใช้ window function ล้วน ๆ ห้าม self-join

ข้อกำหนด
1. YTD ต้อง `PARTITION BY` ปี
2. YoY ต้องใช้ `LAG(..., 12)`
3. Rolling 12 ต้องใช้ `ROWS BETWEEN 11 PRECEDING AND CURRENT ROW`
4. ต้องคืน `NULL` (ไม่ใช่ 0) ในช่วงที่คำนวณไม่ได้

จากนั้นตอบว่า ถ้าบางเดือนไม่มียอดขายเลย (ไม่มีแถวในตาราง)
`ROWS BETWEEN` จะให้ผลผิดอย่างไร และต้องแก้ด้วยอะไร

In [ ]:
SQL = """
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', sales_date) AS ym,
        SUM(net_amount)                 AS amount
    FROM fact_sales_daily
    GROUP BY 1
)
SELECT
    ym,
    amount,

    -- YTD: สะสมภายในปี รีเซ็ตอัตโนมัติด้วย PARTITION BY
    SUM(amount) OVER (
        PARTITION BY EXTRACT(YEAR FROM ym)
        ORDER BY ym
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS ytd,

    -- MoM: NULL อัตโนมัติในเดือนแรก เพราะ LAG ไม่มีค่า
    (amount / NULLIF(LAG(amount, 1) OVER (ORDER BY ym), 0) - 1) * 100 AS mom_pct,

    -- YoY: NULL อัตโนมัติใน 12 เดือนแรก
    (amount / NULLIF(LAG(amount, 12) OVER (ORDER BY ym), 0) - 1) * 100 AS yoy_pct,

    -- Rolling 12: บังคับให้เป็น NULL จนกว่าจะมีครบ 12 เดือน
    CASE WHEN COUNT(*) OVER (ORDER BY ym ROWS BETWEEN 11 PRECEDING AND CURRENT ROW) = 12
         THEN SUM(amount) OVER (ORDER BY ym ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)
    END AS rolling_12m

FROM monthly
ORDER BY ym;
"""
print(SQL)

print("""คำตอบข้อสุดท้าย — เดือนที่ไม่มีแถว
------------------------------------
ROWS BETWEEN 11 PRECEDING นับ 'จำนวนแถว' ไม่ใช่ 'จำนวนเดือน'
ถ้ามีเดือนที่ไม่มียอดขายเลยจนไม่มีแถวในตาราง หน้าต่างจะกวาดย้อนไปไกลเกิน 12 เดือนจริง
Rolling 12 จึงกลายเป็นผลรวมของ 13-14 เดือนโดยไม่มีสัญญาณเตือนใด ๆ

วิธีแก้
  ก. ใช้ dim_date เป็นแกนหลักแล้ว LEFT JOIN ยอดขายเข้าไป
     เพื่อรับประกันว่าทุกเดือนมีแถวเสมอ (ยอดเป็น 0 ในเดือนที่ไม่มีการขายจริง)
  ข. หรือเปลี่ยนไปใช้ RANGE BETWEEN INTERVAL '11 months' PRECEDING
     ซึ่งนับตามค่าของคอลัมน์เวลาแทนจำนวนแถว (รองรับต่างกันในแต่ละฐานข้อมูล)

ทางเลือก ก. เป็นมาตรฐานของคลังข้อมูล เพราะแก้ปัญหานี้ให้ทุกตัววัดพร้อมกัน
ไม่ใช่แก้ทีละคำสั่ง — และเป็นเหตุผลหลักที่ dim_date ต้องมีทุกวันครบถ้วนเสมอ
""")

---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — ยุบระดับเดือนถูกต้องและเรียงตามเวลา | 1 |
| งานที่ 2 — ตัววัดทั้ง 4 ถูกต้องและ NaN ครบตามที่ควรเป็น | 3 |
| งานที่ 3 — หาเหตุการณ์ผิดปกติและระบุช่วงวันได้ | 3 |
| งานที่ 4 — เปรียบเทียบความสะดุดตาข้ามตัววัดพร้อมคำอธิบาย | 3 |
| งานที่ 5 — อธิบายผลย้อนกลับของ YoY และเสนอวิธีแก้ | 3 |
| งานที่ 6 — แสดงผลของการเติม 0 แทน NaN ด้วยตัวเลข | 2 |
| งานที่ 7 — SQL window function ครบเงื่อนไข + ปัญหาเดือนที่ไม่มีแถว | 3 |
| **รวม** | **18** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/time-intelligence`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง